# Radiant RAG MCP — Video Ingestion & Summarization Test Notebook

End-to-end test of the `ingest_video` MCP tool and `VideoSummarizationAgent`.
Synthetic test videos are generated in-process so the suite runs without any
external media files.

| | |
|---|---|
| **Transport** | Streamable HTTP |
| **Backend** | ChromaDB |
| **LLM** | Ollama Cloud — set `OLLAMA_API_KEY` in Colab Secrets |
| **VLM** | `Qwen/Qwen2-VL-2B-Instruct` (T4 GPU recommended) |

| # | Section | LLM | VLM |
|---|---------|-----|-----|
| 1 | Install video extras | | |
| 2 | Import checks | | |
| 3 | Configuration display | | |
| 4 | Shared helpers (client, pretty-print) | | |
| 5 | Server startup | | |
| 6 | Tool list verification | | |
| 7 | Create synthetic test videos | | |
| 8 | `_has_audio()` smoke test | | |
| 9 | `ingest_video` — audio path | | |
| 10 | `ingest_video` — silent path | | VLM |
| 11 | `search_knowledge` across video content | | |
| 12 | `query_knowledge` grounded in video | LLM | |
| 13 | `VideoSummarizationAgent` — `brief` | LLM | |
| 14 | `VideoSummarizationAgent` — `standard` | LLM | |
| 15 | `VideoSummarizationAgent` — `detailed` | LLM | |
| 16 | Word-count comparison (window / chapter / overall) | LLM | |
| 17 | YouTube URL ingestion | LLM | |
| 18 | Cleanup | | |

## 1  Install

Installs the core `radiant-rag-mcp` package and all video dependencies,
then resolves the two distribution gaps that cause `ingest_video` to be absent.

**The video source files do not exist on `main`.**
Set `VIDEO_BRANCH` below to the branch where `video_processor.py` and
`video_summarization.py` live (e.g. `"feature/video"`, `"dev"`, `"video-support"`).
The cell probes a list of candidate branches automatically, so if you leave
`VIDEO_BRANCH = ""` it will try them all and use the first one that has the files.


In [ ]:
import subprocess, sys

CONFIG_PATH = '/content/config.yaml'

# VIDEO_BRANCH: leave "" to use main (post-merge default).
# Set to a branch name only if you are testing a pre-merge feature branch.
VIDEO_BRANCH = ""

# ── Step 1: Pin numpy BEFORE any other install ────────────────────────────────
# Must be first. --force-reinstall on downstream packages ignores pins set
# later in the same cell, so we lock numpy here before anything can upgrade it.
# numpy 2.2+ broke scipy's array_api_compat (_center import error).
!pip install -q --force-reinstall "numpy>=1.26,<2.2"

# ── Step 2: Core RAG package ──────────────────────────────────────────────────
# --upgrade --no-cache-dir: pulls current git HEAD without force-reinstalling
# all transitive deps (which would defeat the numpy pin above).
!pip install -q --upgrade --no-cache-dir "radiant-rag-mcp[chroma] @ git+https://github.com/dshipley71/radiant-rag-mcp.git"
!pip install -q --prefer-binary nest_asyncio httpx "fastmcp>=3.0"
!wget -q "https://raw.githubusercontent.com/dshipley71/radiant-rag-mcp/main/config.yaml" -O {CONFIG_PATH}

# ── Step 3: Reinstall numpy-dependent stack against the pin ───────────────────
# Reinstall scipy, scikit-learn, and sentence-transformers so they are built
# against the pinned numpy, not whatever the previous environment had.
!pip install -q --force-reinstall "numpy>=1.26,<2.2" "scipy>=1.11" "scikit-learn>=1.3" "sentence-transformers>=2.7"

# ── Step 4: Video runtime dependencies ───────────────────────────────────────
!pip install -q yt-dlp "faster-whisper>=1.0.0" "scenedetect[opencv]>=0.6.3"
!pip install -q ffmpeg-python Pillow opencv-python-headless
!pip install -q "moviepy>=1.0.3"
!pip install -q "transformers>=4.57.0"
!pip install -q "accelerate>=0.27.0"
!pip install -q "qwen-vl-utils" 2>/dev/null || true
!apt-get install -qq -y ffmpeg > /dev/null 2>&1
!pip install -q fasttext-wheel 2>/dev/null || true

# ── Step 5: Verify numpy version before importing anything ────────────────────
import importlib, subprocess as _sp
_np_ver = _sp.run(
    [sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
    capture_output=True, text=True
).stdout.strip()
print(f'numpy version in subprocess : {_np_ver}')
_major, _minor = (int(x) for x in _np_ver.split('.')[:2])
if (_major, _minor) >= (2, 2):
    raise RuntimeError(
        f'numpy {_np_ver} is still active after pinning — '
        'restart the Colab runtime (Runtime > Restart session) '
        'and re-run this cell.'
    )
print(f'numpy {_np_ver}  OK  (< 2.2)')

# ── Kernel restart (REQUIRED after numpy downgrade) ───────────────────────────
# pip has written numpy 2.1.x to disk, but the running kernel still has the
# old numpy loaded in memory. A restart is mandatory; Colab will reconnect
# automatically. Continue from cell 1b after the kernel comes back up.
print('Installs complete. Restarting kernel to activate pinned numpy ...')
import time; time.sleep(1)
import os; os.kill(os.getpid(), 9)   # SIGKILL — Colab restarts automatically


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 100.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.1.3 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 235.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 294.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 417.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 393.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build whee

### ↻ Kernel restarted — continue from here

The kernel was restarted to activate the pinned `numpy<2.2`.
Run the cell below to cache the embedding models and fetch the video submodule files.


In [1]:
# Runs after the kernel restart triggered by cell 1a.
# numpy is now the on-disk version (<2.2); safe to import sentence_transformers.
import warnings, subprocess, sys
warnings.filterwarnings('ignore', category=SyntaxWarning, module='moviepy')

# Confirm numpy version in the live kernel
import numpy as np
print(f'numpy (live kernel) : {np.__version__}')
assert tuple(int(x) for x in np.__version__.split('.')[:2]) < (2, 2), (
    f'numpy {np.__version__} is still >=2.2 — '
    'try Runtime > Restart session manually, then re-run from cell 1b.'
)

# ── Fix torchvision / PyTorch CUDA version mismatch ─────────────────────────
# --force-reinstall can pull torchvision built for a different CUDA major than
# the installed PyTorch, causing: RuntimeError: compiled with different CUDA
# major versions. Detect the live torch CUDA tag and reinstall to match.
import subprocess as _sp, re as _re
_torch_info = _sp.run(
    [sys.executable, '-c', 'import torch; print(torch.__version__, torch.version.cuda)'],
    capture_output=True, text=True
).stdout.strip().split()
_torch_ver  = _torch_info[0] if _torch_info else 'unknown'
_torch_cuda = _torch_info[1] if len(_torch_info) > 1 else None
print(f'torch version : {_torch_ver}   CUDA : {_torch_cuda}')

if _torch_cuda:
    _major = _torch_cuda.split('.')[0]           # e.g. "13" from "13.0"
    _cu_tag = f'cu{_major}0'                     # e.g. "cu130"
    _whl = f'https://download.pytorch.org/whl/{_cu_tag}'
    print(f'Reinstalling torchvision from {_whl} ...')
    _r = _sp.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         '--force-reinstall', f'torchvision',
         '--index-url', _whl],
        capture_output=True, text=True
    )
    if _r.returncode == 0:
        print(f'torchvision reinstalled for {_cu_tag}  OK')
    else:
        # Wheel index miss (e.g. cu130 not published yet) — fall back to PyPI
        print(f'  {_cu_tag} wheel not found, falling back to PyPI torchvision ...')
        _sp.run([sys.executable, '-m', 'pip', 'install', '-q',
                 '--force-reinstall', 'torchvision'], check=True)
        print('torchvision reinstalled from PyPI  OK')
else:
    print('Could not detect CUDA version, skipping torchvision fix.')

# ── Step 6: Pre-cache embedding / reranking models ───────────────────────────
import warnings
warnings.filterwarnings('ignore', category=SyntaxWarning, module='moviepy')
from sentence_transformers import SentenceTransformer, CrossEncoder
SentenceTransformer('sentence-transformers/all-MiniLM-L12-v2')
print('sentence-transformers  OK')
CrossEncoder('cross-encoder/ms-marco-MiniLM-L12-v2')
print('cross-encoder          OK')

# ── Step 7: Fetch video submodule files from the correct branch ───────────────
import importlib.util, json as _json, urllib.request
from pathlib import Path

REPO  = 'dshipley71/radiant-rag-mcp'
HEADS = 'https://api.github.com/repos/{repo}/branches?per_page=100'

_VIDEO_FILES = [
    'radiant_rag_mcp/ingestion/video_processor.py',
    'radiant_rag_mcp/agents/video_summarization.py',
]

def _raw_url(branch, path):
    return f'https://raw.githubusercontent.com/{REPO}/{branch}/{path}'

def _probe_url(url):
    try:
        req = urllib.request.Request(url, method='HEAD')
        with urllib.request.urlopen(req, timeout=6) as r:
            return r.status == 200
    except Exception:
        return False

def _fetch_url(url, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, dest)
        return True
    except Exception:
        import subprocess as _sp
        r = _sp.run(['wget', '-q', url, '-O', str(dest)], capture_output=True)
        if r.returncode != 0:
            dest.unlink(missing_ok=True)
        return r.returncode == 0

def _pkg_root(pkg):
    spec = importlib.util.find_spec(pkg)
    if spec is None:
        raise ImportError(f'Cannot locate installed package: {pkg}')
    return Path(spec.origin).parent

_CANDIDATES = [
    'feature/video', 'feature/video-support', 'feature/video-ingestion',
    'video', 'video-support', 'video-ingestion',
    'dev', 'develop', 'development',
    'feat/video', 'feat/video-support',
    'video-summarization', 'video-processing',
]

try:
    _root = _pkg_root('radiant_rag_mcp')
    print(f'\nPackage root : {_root}')
    _branch = VIDEO_BRANCH.strip() or None

    if _branch:
        print(f'Using specified branch: {_branch}')
        _branches_to_try = [_branch]
    else:
        print('Fetching branch list from GitHub API ...')
        try:
            with urllib.request.urlopen(HEADS.format(repo=REPO), timeout=10) as _r:
                _all_branches = [b['name'] for b in _json.loads(_r.read())]
            print(f'  Repo branches: {_all_branches}')
            _branches_to_try = (
                [b for b in _CANDIDATES if b in _all_branches]
                + [b for b in _all_branches if b not in _CANDIDATES and b != 'main']
            ) or _all_branches
        except Exception as e:
            print(f'  GitHub API unavailable ({e}); trying candidate list blind.')
            _branches_to_try = _CANDIDATES

    _probe_file = _VIDEO_FILES[0]
    _found_branch = None
    print(f'\nProbing branches for {_probe_file} ...')
    for _b in _branches_to_try:
        if _probe_url(_raw_url(_b, _probe_file)):
            _found_branch = _b
            print(f'  FOUND on branch: {_b}')
            break
        else:
            print(f'  not on: {_b}')

    if _found_branch is None:
        print()
        print('[WARNING] Could not locate video files on any probed branch.')
        print('  Set VIDEO_BRANCH to the correct branch name and re-run.')
        print('  Branches tried:', _branches_to_try)
        print('  (Files may already be present from main after the merge.)')
    else:
        print(f'\nFetching video files from branch: {_found_branch}')
        _raw_base = f'https://raw.githubusercontent.com/{REPO}/{_found_branch}'
        try:
            with urllib.request.urlopen(
                    f'https://api.github.com/repos/{REPO}/git/trees/{_found_branch}?recursive=1',
                    timeout=10) as _r:
                _extra = [
                    item['path'] for item in _json.loads(_r.read()).get('tree', [])
                    if item['type'] == 'blob'
                    and item['path'].endswith('.py')
                    and 'video' in item['path'].lower()
                    and item['path'].startswith('radiant_rag_mcp/')
                ]
            if _extra:
                print(f'  Additional video files on {_found_branch}: {_extra}')
            _to_fetch = list(dict.fromkeys(_VIDEO_FILES + _extra))
        except Exception:
            _to_fetch = _VIDEO_FILES

        for repo_path in _to_fetch:
            dest = _root / repo_path[len('radiant_rag_mcp/'):]
            if dest.exists():
                print(f'  already present  {repo_path}')
            else:
                url = f'{_raw_base}/{repo_path}'
                print(f'  fetching  {repo_path}')
                print(f'    {"installed" if _fetch_url(url, dest) else "FETCH FAILED"} -> {dest}')

        _server_py = _root / 'server.py'
        if _server_py.exists():
            _srv = _server_py.read_text()
            for vf in sorted(_root.rglob('*video*.py')):
                if vf == _server_py:
                    continue
                _content = vf.read_text()
                if 'ingest_video' in _content or '@mcp.tool' in _content:
                    _mod = str(vf.relative_to(_root.parent)).replace('/', '.').replace('\\', '.')[:-3]
                    if _mod not in _srv and 'ingest_video' not in _srv:
                        print(f'\nPatching server.py: adding import {_mod}')
                        _server_py.write_text(f'import {_mod}  # video tool\n' + _srv)
                    else:
                        print(f'  server.py already references {_mod}')

        print(f'\nVideo fetch complete (branch: {_found_branch}).')
        print(f'Set VIDEO_BRANCH = "{_found_branch}" to skip probing next run.')

except Exception as e:
    import traceback
    print(f'[ERROR] {e}')
    traceback.print_exc()

print('\nInstall complete')


numpy (live kernel) : 2.1.3
torch version : 2.11.0+cu130   CUDA : 13.0
Reinstalling torchvision from https://download.pytorch.org/whl/cu130 ...
torchvision reinstalled for cu130  OK


ImportError: cannot import name '_slice' from 'numpy._core.umath' (/usr/local/lib/python3.12/dist-packages/numpy/_core/umath.py)

## 2  Import checks

Verifies every dependency the video pipeline needs is importable.
A `MISSING` line means the package failed to install in section 1.

In [ ]:
import sys
import importlib
import traceback
from pathlib import Path

# ── Required third-party packages (hard-fail) ─────────────────────────────────
REQUIRED = [
    ('fastmcp',               'fastmcp'),
    ('nest_asyncio',          'nest_asyncio'),
    ('chromadb',              'chromadb'),
    ('sentence_transformers', 'sentence-transformers'),
    ('yt_dlp',                'yt-dlp'),
    ('faster_whisper',        'faster-whisper'),
    ('cv2',                   'opencv-python-headless'),
    ('PIL',                   'Pillow'),
    ('scenedetect',           'scenedetect[opencv]'),
    ('ffmpeg',                'ffmpeg-python'),
    ('transformers',          'transformers'),
]

# ── Radiant video submodules (soft-fail with full traceback diagnostic) ────────
VIDEO_CORE = [
    ('radiant_rag_mcp',                            'radiant-rag-mcp'),
    ('radiant_rag_mcp.ingestion.video_processor',  'see diagnostic below'),
    ('radiant_rag_mcp.agents.video_summarization', 'see diagnostic below'),
]

_missing_required = []
_missing_video    = {}   # module -> exception

print('=== Required packages ===')
for module, pkg in REQUIRED:
    try:
        __import__(module)
        print(f'  ✓       {module}')
    except ImportError as e:
        print(f'  ✗  {module}  ->  pip install {pkg}')
        _missing_required.append(pkg)

print()
print('=== Radiant RAG video submodules ===')
for module, _ in VIDEO_CORE:
    try:
        __import__(module)
        print(f'  ✓       {module}')
    except Exception as e:
        print(f'  ✗     {module}')
        _missing_video[module] = e

# ── Diagnostic: print the real import error for each failing submodule ─────────
if _missing_video:
    print()
    print('  ─── Import error details ───────────────────────────────────────────')
    for mod, exc in _missing_video.items():
        print(f'  Module : {mod}')
        print(f'  Error  : {type(exc).__name__}: {exc}')
        # Walk to the root cause (ImportError chain)
        cause = exc.__cause__ or exc.__context__
        if cause:
            print(f'  Caused by: {type(cause).__name__}: {cause}')
        print()

    # Show which files exist on disk so we know if it's missing vs broken
    try:
        import radiant_rag_mcp as _pkg
        _root = Path(_pkg.__file__).parent
        print(f'  Package root: {_root}')
        for subpath in ['ingestion', 'ingestion/video_processor.py',
                        'agents',   'agents/video_summarization.py']:
            p = _root / subpath
            print(f'  {"exists" if p.exists() else "ABSENT "}  {p}')
    except Exception as probe_err:
        print(f'  Package probe failed: {probe_err}')

    print()
    print('  ► The most common root cause is a missing top-level import inside')
    print('    video_processor.py or video_summarization.py (e.g. moviepy,')
    print('    transformers, faster_whisper).  The error line above identifies it.')
    print('  ► Install the missing package, then restart the runtime and re-run')
    print('    sections 1 and 2.')

if _missing_required:
    raise ImportError(f'Missing required packages: {", ".join(_missing_required)}')

if _missing_video:
    import warnings
    warnings.warn(
        f'Video submodules not importable: {list(_missing_video)}.  '
        'Sections 8-18 will fail.  See diagnostic above.',
        stacklevel=1,
    )
else:
    print()
    print('All imports ok')


=== Required packages ===
  ✓       fastmcp
  ✓       nest_asyncio
  ✓       chromadb
  ✓       sentence_transformers
  ✓       yt_dlp
  ✓       faster_whisper
  ✓       cv2
  ✓       PIL


/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


  ✓       scenedetect
  ✓       ffmpeg
  ✓       transformers

=== Radiant RAG video submodules ===
  ✓       radiant_rag_mcp


  warnings.warn(



  ✗     radiant_rag_mcp.ingestion.video_processor
  ✗     radiant_rag_mcp.agents.video_summarization

  ─── Import error details ───────────────────────────────────────────
  Module : radiant_rag_mcp.ingestion.video_processor
  Error  : ModuleNotFoundError: No module named 'radiant_rag_mcp.ingestion.video_processor'

  Module : radiant_rag_mcp.agents.video_summarization
  Error  : ModuleNotFoundError: No module named 'radiant_rag_mcp.agents.video_summarization'

  Package root: /usr/local/lib/python3.12/dist-packages/radiant_rag_mcp
  exists  /usr/local/lib/python3.12/dist-packages/radiant_rag_mcp/ingestion
  ABSENT   /usr/local/lib/python3.12/dist-packages/radiant_rag_mcp/ingestion/video_processor.py
  exists  /usr/local/lib/python3.12/dist-packages/radiant_rag_mcp/agents
  ABSENT   /usr/local/lib/python3.12/dist-packages/radiant_rag_mcp/agents/video_summarization.py

  ► The most common root cause is a missing top-level import inside
    video_processor.py or video_summarization.py (

## 3  Configuration

Sets every environment variable the server needs before it starts.
Video-specific keys use the prefix `RADIANT_VIDEO_*`.
LLM and VLM settings are displayed so they are easy to spot-check.

> Add `OLLAMA_API_KEY` to **Colab Secrets** (key icon in the left sidebar)
> before running `[LLM]` cells.

In [ ]:
import os

# Read LLM key from Colab Secrets, falling back to env
try:
    from google.colab import userdata
    _key = userdata.get('OLLAMA_API_KEY') or ''
except Exception:
    _key = os.environ.get('RADIANT_OLLAMA_API_KEY', '')

# Core server settings
os.environ['RADIANT_OLLAMA_BASE_URL']  = 'https://ollama.com/v1'
os.environ['RADIANT_OLLAMA_API_KEY']   = _key
os.environ['RADIANT_TRANSPORT']        = 'http'
os.environ['RADIANT_HOST']             = '127.0.0.1'
os.environ['RADIANT_PORT']             = '8765'
os.environ['RADIANT_STORAGE_BACKEND']  = 'chroma'
os.environ['RADIANT_CONFIG_PATH']      = CONFIG_PATH

# Pipeline flags
os.environ['RADIANT_PIPELINE_USE_CRITIC']       = 'false'
os.environ['RADIANT_CITATION_ENABLED']          = 'false'
os.environ['RADIANT_LLM_BACKEND_TIMEOUT']       = '120'
os.environ['RADIANT_LLM_BACKEND_MAX_RETRIES']   = '0'
os.environ['RADIANT_RERANKING_BACKEND_DEVICE']  = 'cuda'
os.environ['RADIANT_EMBEDDING_BACKEND_DEVICE']  = 'cuda'
os.environ['RADIANT_CHUNKING_USE_LLM_CHUNKING'] = 'false'
os.environ['RADIANT_RETRIEVAL_DENSE_TOP_K']     = '5'
os.environ['RADIANT_RETRIEVAL_BM25_TOP_K']      = '5'

# VLM — required for silent video frame-window captioning
os.environ['RADIANT_VLM_ENABLED']    = 'true'
os.environ['RADIANT_VLM_MODEL_NAME'] = 'Qwen/Qwen2-VL-2B-Instruct'
os.environ['RADIANT_VLM_DEVICE']     = 'auto'

# Video processor settings
os.environ['RADIANT_VIDEO_WHISPER_MODEL']                 = 'base'
os.environ['RADIANT_VIDEO_WINDOW_DURATION_SECONDS']       = '10.0'
os.environ['RADIANT_VIDEO_WINDOW_OVERLAP_SECONDS']        = '2.0'
os.environ['RADIANT_VIDEO_FRAMES_PER_WINDOW']             = '3'
os.environ['RADIANT_VIDEO_ENABLE_SCENE_CHANGE_DETECTION'] = 'true'
os.environ['RADIANT_VIDEO_SCENE_CHANGE_THRESHOLD']        = '0.25'
os.environ['RADIANT_VIDEO_FILMSTRIP_TILE_WIDTH']          = '480'
os.environ['RADIANT_VIDEO_FILMSTRIP_TILE_HEIGHT']         = '270'
os.environ['RADIANT_VIDEO_ENABLE_SILENT_VIDEO_ANALYSIS']  = 'true'

# Summarization defaults (overridden per-cell in sections 13-16)
os.environ['RADIANT_VIDEO_SUMMARIZATION_SUMMARY_DETAIL']           = 'standard'
os.environ['RADIANT_VIDEO_SUMMARIZATION_WINDOW_CAPTION_SENTENCES'] = '4'

SERVER_URL  = f"http://{os.environ['RADIANT_HOST']}:{os.environ['RADIANT_PORT']}/mcp"
HAS_LLM_KEY = bool(_key)

# Display configuration summary
print('=== Configuration ===')
print(f'  Server URL      : {SERVER_URL}')
print(f'  LLM key set     : {HAS_LLM_KEY}')
print(f'  VLM model       : {os.environ["RADIANT_VLM_MODEL_NAME"]}')
print(f'  Whisper model   : {os.environ["RADIANT_VIDEO_WHISPER_MODEL"]}')
print(f'  Window duration : {os.environ["RADIANT_VIDEO_WINDOW_DURATION_SECONDS"]}s')
print(f'  Window overlap  : {os.environ["RADIANT_VIDEO_WINDOW_OVERLAP_SECONDS"]}s')
print(f'  Frames/window   : {os.environ["RADIANT_VIDEO_FRAMES_PER_WINDOW"]}')
print(f'  Summary detail  : {os.environ["RADIANT_VIDEO_SUMMARIZATION_SUMMARY_DETAIL"]}')
print(f'  Scene detection : {os.environ["RADIANT_VIDEO_ENABLE_SCENE_CHANGE_DETECTION"]}')

if not HAS_LLM_KEY:
    print()
    print('[NOTE] OLLAMA_API_KEY not found.')
    print('       Add it to Colab Secrets (key icon, left sidebar).')
    print('       Cells tagged [LLM] will be skipped automatically.')

## 4  Shared helpers

Defines:
- `run(tool, args)` — call an MCP tool and pretty-print the JSON result
- `assert_ok(result)` — raise `AssertionError` on tool errors
- `skip_llm()` — guard used in every `[LLM]` cell
- `_wait_for_server()` — async HTTP probe used by section 5

In [ ]:
import asyncio
import json
import logging
import threading
import time
import textwrap
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()  # Allow asyncio.run() inside Jupyter/Colab event loop

import httpx as _httpx
from fastmcp import Client
from fastmcp.exceptions import ToolError


async def _call(tool: str, args: dict = None):
    """Async helper: open a short-lived MCP client, call the tool, return parsed result."""
    try:
        async with Client(SERVER_URL) as client:
            raw = await client.call_tool(tool, args or {})
    except ToolError as e:
        return {'status': 'error', 'tool': tool, 'message': str(e)}
    except Exception as e:
        return {'status': 'error', 'tool': tool, 'message': f'{type(e).__name__}: {e}'}

    content = raw.content if hasattr(raw, 'content') else (raw or [])
    if not content:
        return None
    item = content[0]
    text = item.text if hasattr(item, 'text') else str(item)
    try:
        return json.loads(text)
    except (json.JSONDecodeError, ValueError):
        return text


def run(tool: str, args: dict = None):
    """Synchronous wrapper around _call; pretty-prints and returns the result."""
    result = asyncio.run(_call(tool, args))
    print(json.dumps(result, indent=2, default=str))
    return result


def skip_llm() -> bool:
    """Return True (and print a notice) when no LLM key is available."""
    if not HAS_LLM_KEY:
        print('[SKIPPED] Set OLLAMA_API_KEY in Colab Secrets to run this cell.')
        return True
    return False


def assert_ok(result, field: str = None):
    """Assert the tool result is not an error dict; optionally check a field exists."""
    assert result is not None, 'Tool returned no result'
    if isinstance(result, dict):
        if result.get('status') == 'error':
            raise AssertionError(f'Tool error: {result.get("message", result)}')
        if field:
            assert field in result, f'Expected field "{field}" missing from result: {result}'
    print('[OK]')


async def _wait_for_server(url: str, timeout: int = 120, interval: float = 1.0) -> bool:
    """Poll the MCP endpoint until it responds or the timeout expires."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            async with _httpx.AsyncClient() as http:
                await http.get(url, timeout=2.0,
                               headers={'Accept': 'application/json, text/event-stream'})
                return True
        except (_httpx.ConnectError, _httpx.ConnectTimeout, _httpx.ReadTimeout):
            await asyncio.sleep(interval)
    return False


print('Helpers loaded')

## 5  Server startup

Starts the Radiant RAG MCP server in a background daemon thread.
Tries `http` then `streamable-http` transport automatically.
Polls the endpoint for up to 90 s before raising `TimeoutError`.

In [ ]:
import subprocess, time as _time

# Kill any process already bound to the port so restarts work cleanly
_port = int(os.environ['RADIANT_PORT'])
subprocess.run(['fuser', '-k', f'{_port}/tcp'], capture_output=True)
_time.sleep(1.0)

logging.basicConfig(stream=__import__('sys').stderr, level=logging.WARNING, force=True)

_server_ready   = threading.Event()
_server_error   = None
_transport_used = None


def _run_server():
    """Target for the daemon thread that hosts the MCP server."""
    global _server_error, _transport_used
    try:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        from radiant_rag_mcp.server import mcp
        print('Package  : radiant_rag_mcp  ok')
        _server_ready.set()
        host = os.environ['RADIANT_HOST']
        port = int(os.environ['RADIANT_PORT'])
        for _t in ['http', 'streamable-http']:
            try:
                _transport_used = _t
                mcp.run(transport=_t, host=host, port=port)
                return
            except Exception as _e:
                if 'Unknown transport' in str(_e) or 'unknown transport' in str(_e).lower():
                    continue
                raise
        raise RuntimeError('No supported HTTP transport found (tried: http, streamable-http)')
    except Exception as exc:
        _server_error = exc
        if not _server_ready.is_set():
            _server_ready.set()


_thread = threading.Thread(target=_run_server, name='radiant-mcp', daemon=True)
_thread.start()

# Wait for the import to complete before polling the HTTP endpoint
if not _server_ready.wait(timeout=30):
    raise TimeoutError('Server thread did not signal ready within 30 s.')
if _server_error:
    raise _server_error

print(f'Waiting for server at {SERVER_URL} ...')
_deadline = time.time() + 90
while time.time() < _deadline:
    if _server_error:
        raise RuntimeError(f'Server thread failed: {_server_error}')
    if asyncio.run(_wait_for_server(SERVER_URL, timeout=5)):
        print(f'Server ready  ->  {SERVER_URL}  (transport: {_transport_used})')
        break
    time.sleep(1)
else:
    raise TimeoutError('Server did not bind within 90 s.')

## 6  Tool list verification

Lists every registered MCP tool and asserts that `ingest_video` is present.
A missing tool means the video sub-package did not install or register correctly.

In [ ]:
# Retrieve the tool manifest from the live server
async def _list_tools():
    async with Client(SERVER_URL) as client:
        return await client.list_tools()

tools      = asyncio.run(_list_tools())
registered = {t.name for t in tools}

print(f'{len(tools)} tool(s) registered:')
for t in sorted(tools, key=lambda x: x.name):
    desc = (t.description or '').splitlines()[0]
    print(f'  {t.name:<30}  {desc}')

# ── ingest_video assertion with actionable diagnostic ────────────────────────
if 'ingest_video' in registered:
    print()
    print('[ASSERT OK] ingest_video is registered')
else:
    import importlib.util
    from pathlib import Path

    print()
    print('[FAIL] ingest_video is NOT in the tool list.')
    print()
    print('Diagnostic:')

    # Locate package root
    spec = importlib.util.find_spec('radiant_rag_mcp')
    if spec:
        root = Path(spec.origin).parent
        print(f'  Package root : {root}')

        # Check which video files exist on disk
        video_files = sorted(root.rglob('*video*.py'))
        if video_files:
            print(f'  Video .py files on disk:')
            for vf in video_files:
                has_tool = 'ingest_video' in vf.read_text()
                marker   = '  <-- ingest_video defined here' if has_tool else ''
                print(f'    {vf.relative_to(root)}{marker}')
        else:
            print('  No video .py files found — section 1 fetch may have failed.')

        # Check server.py for video imports
        server_py = root / 'server.py'
        if server_py.exists():
            srv = server_py.read_text()
            video_imports = [ln.strip() for ln in srv.splitlines()
                             if 'video' in ln.lower() and ln.strip().startswith('import')]
            if video_imports:
                print(f'  server.py video imports : {video_imports}')
            else:
                print('  server.py has NO video imports — patch in section 1 may have failed.')
        else:
            print('  server.py not found.')
    else:
        print('  radiant_rag_mcp not importable — check section 1.')

    print()
    print('  ► Re-run section 1, then restart the runtime and re-run sections 1-6.')
    print('  ► If the file containing ingest_video does not exist in the repo yet,')
    print('    the feature branch name is needed to point RAW_BASE at the right ref.')
    print()
    raise AssertionError(
        'ingest_video not registered after fetch/patch. See diagnostic above.'
    )


## 7  Create synthetic test videos

Generates two 30-second, 640×480 MP4 files entirely in-process:

| File | Content | Audio |
|------|---------|-------|
| `audio_test.mp4` | SMPTE-style colour bars (6 bands) | 1 kHz sine tone, AAC |
| `silent_test.mp4` | Random solid-colour frames per second | None |

The colour bars give `VideoProcessor` meaningful scene-change boundaries.
The silent video exercises the VLM frame-captioning path in section 10.

In [ ]:
import cv2
import numpy as np
import subprocess
from pathlib import Path

VIDEO_DIR    = Path('/tmp/radiant_video_test')
VIDEO_DIR.mkdir(exist_ok=True)
AUDIO_TEST   = str(VIDEO_DIR / 'audio_test.mp4')
SILENT_TEST  = str(VIDEO_DIR / 'silent_test.mp4')

FPS = 10
DUR = 30   # seconds
W, H = 640, 480  # 640×480 as specified


# --- Colour-bar generator (SMPTE-style, 6 vertical bands) ---
def _colour_bar_frame(frame_idx: int, total_frames: int) -> np.ndarray:
    """
    Return a 640×480 BGR frame with 6 SMPTE-style colour bars.
    A thin progress ticker at the bottom moves with frame_idx.
    """
    # White, Yellow, Cyan, Green, Magenta, Red, Blue (classic 7-bar; use 6)
    BAR_COLOURS = [
        (235, 235, 235),  # White
        (  0, 235, 235),  # Yellow  (BGR)
        (235, 235,   0),  # Cyan
        (  0, 235,   0),  # Green
        (235,   0, 235),  # Magenta
        (  0,   0, 235),  # Red
    ]
    frame     = np.zeros((H, W, 3), dtype=np.uint8)
    bar_w     = W // len(BAR_COLOURS)
    bar_h     = int(H * 0.75)  # bars fill top 75 %
    for i, col in enumerate(BAR_COLOURS):
        x0 = i * bar_w
        x1 = x0 + bar_w if i < len(BAR_COLOURS) - 1 else W
        frame[:bar_h, x0:x1] = col
    # Grey ramp in the lower 25 %
    for x in range(W):
        g = int((x / W) * 235)
        frame[bar_h:, x] = (g, g, g)
    # Timecode overlay
    t_sec = frame_idx / FPS
    cv2.putText(frame, f't={t_sec:.1f}s', (12, H - 12),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    # Progress ticker
    px = int((frame_idx / max(1, total_frames - 1)) * (W - 10)) + 5
    cv2.circle(frame, (px, H - 30), 5, (0, 200, 255), -1)
    return frame


def _write_raw_colourbar(path: str) -> None:
    """Write a raw (no re-encode) colour-bar video using mp4v codec."""
    total   = DUR * FPS
    fourcc  = cv2.VideoWriter_fourcc(*'mp4v')
    writer  = cv2.VideoWriter(path, fourcc, FPS, (W, H))
    for fi in range(total):
        writer.write(_colour_bar_frame(fi, total))
    writer.release()


# --- Random-colour generator for silent video ---
def _write_raw_random(path: str) -> None:
    """
    Write a raw video where each second uses a new random solid colour.
    This ensures every 1-second block is visually distinct for the VLM.
    """
    rng     = np.random.default_rng(seed=42)
    total   = DUR * FPS
    fourcc  = cv2.VideoWriter_fourcc(*'mp4v')
    writer  = cv2.VideoWriter(path, fourcc, FPS, (W, H))
    for fi in range(total):
        # New colour every second (FPS frames)
        if fi % FPS == 0:
            _col = tuple(int(c) for c in rng.integers(30, 220, size=3))
        frame = np.full((H, W, 3), _col, dtype=np.uint8)
        t_sec = fi / FPS
        cv2.putText(frame, f't={t_sec:.1f}s', (12, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        writer.write(frame)
    writer.release()


# ---- Build audio_test.mp4 (colour bars + 1 kHz sine tone) ----
_raw_a = str(VIDEO_DIR / '_raw_audio.mp4')
_wav   = str(VIDEO_DIR / '_sine_1k.wav')

print('Generating colour-bar frames ...')
_write_raw_colourbar(_raw_a)

print('Generating 1 kHz sine tone ...')
subprocess.run(
    ['ffmpeg', '-y', '-f', 'lavfi',
     '-i', f'sine=frequency=1000:duration={DUR}',   # 1 kHz as specified
     _wav],
    capture_output=True, check=True
)

print('Muxing audio_test.mp4 ...')
subprocess.run(
    ['ffmpeg', '-y', '-i', _raw_a, '-i', _wav,
     '-c:v', 'libx264', '-crf', '28',
     '-c:a', 'aac', '-shortest', AUDIO_TEST],
    capture_output=True, check=True
)

# ---- Build silent_test.mp4 (random colours, no audio stream) ----
_raw_s = str(VIDEO_DIR / '_raw_silent.mp4')

print('Generating random-colour frames ...')
_write_raw_random(_raw_s)

print('Encoding silent_test.mp4 (no audio stream) ...')
subprocess.run(
    ['ffmpeg', '-y', '-i', _raw_s,
     '-an',                                # explicitly strip any audio
     '-vcodec', 'libx264', '-crf', '28',
     SILENT_TEST],
    capture_output=True, check=True
)

# Remove intermediate files
for _f in [_raw_a, _raw_s, _wav]:
    Path(_f).unlink(missing_ok=True)

print()
print('Test videos created:')
for vp in [AUDIO_TEST, SILENT_TEST]:
    p = Path(vp)
    print(f'  {p.name:<25}  {p.stat().st_size // 1024:>6} KB  ->  {vp}')

assert Path(AUDIO_TEST).exists(),  f'audio_test.mp4 not created: {AUDIO_TEST}'
assert Path(SILENT_TEST).exists(), f'silent_test.mp4 not created: {SILENT_TEST}'
print('[ASSERT OK] Both test videos exist on disk')

## 8  `_has_audio()` smoke test

Calls `VideoProcessor._has_audio()` directly (no MCP round-trip) and asserts:
- `audio_test.mp4` → `True`
- `silent_test.mp4` → `False`

This validates that the ffprobe-backed audio detection works before ingestion.

In [ ]:
from radiant_rag_mcp.ingestion.video_processor import VideoProcessor
from radiant_rag_mcp.config import VideoProcessorConfig

# Instantiate with default config; _has_audio() only uses ffprobe
cfg = VideoProcessorConfig()
vp  = VideoProcessor(config=cfg)

test_cases = [
    (AUDIO_TEST,  True,  'audio_test.mp4  should have audio'),
    (SILENT_TEST, False, 'silent_test.mp4 should have NO audio'),
]

for path, expected, description in test_cases:
    result = vp._has_audio(path)
    status = 'ok' if result == expected else 'MISMATCH'
    print(f'  {status}  {Path(path).name:<25}  _has_audio={result}  (expected {expected})  {description}')
    # Hard assertion — mismatch means ffprobe or the file is broken
    assert result == expected, (
        f'_has_audio mismatch for {Path(path).name}: '
        f'got {result}, expected {expected}'
    )

print()
print('[ASSERT OK] _has_audio() returns correct values for both test files')

## 9  `ingest_video` — audio path  *(Whisper transcription)*

Ingests `audio_test.mp4` via the MCP tool.
Because the file has an audio stream, `VideoProcessor` routes it through
Whisper for transcription.  Asserts:
- `result["audio_sources"] == 1`
- `result["silent_sources"] == 0`

In [ ]:
%%time
# Ingest the audio video; disable frame captioning to keep runtime short
print('=== ingest_video (audio_test.mp4 — Whisper path) ===')
r = run('ingest_video', {
    'sources':                [AUDIO_TEST],
    'hierarchical':           True,
    'enable_frame_captioning': False,
    'force_frame_analysis':    False,
    'summarize':               False,
})

assert_ok(r, 'sources_processed')

# Verify routing: audio file must land on the Whisper path
assert r.get('audio_sources')  == 1, \
    f'Expected audio_sources=1, got {r.get("audio_sources")}'
assert r.get('silent_sources') == 0, \
    f'Expected silent_sources=0, got {r.get("silent_sources")}'

print(f'\nChunks created  : {r.get("chunks_created")}')
print(f'Audio sources   : {r.get("audio_sources")}')
print(f'Silent sources  : {r.get("silent_sources")}')
print(f'Errors          : {r.get("errors")}')
print('[ASSERT OK] audio_sources==1, silent_sources==0')

In [ ]:
# Confirm index stats after audio ingestion
print('=== get_index_stats (after audio ingest) ===')
r = run('get_index_stats')
assert_ok(r)

## 10  `ingest_video` — silent path  *(VLM frame-window analysis)*  `[VLM]`

Ingests `silent_test.mp4`, which has **no audio stream**.
`VideoProcessor` detects the missing audio and routes to
`_process_silent_video()`: samples frames, tiles filmstrips, then captions
each window with the VLM.

Asserts:
- `result["silent_sources"] == 1`
- `result["audio_sources"] == 0`

> **Runtime:** ~2–5 min on T4 GPU (30 s video ÷ 10 s windows = 3 windows).

In [ ]:
%%time
# Ingest the silent video — triggers VLM frame-captioning path
print('=== ingest_video (silent_test.mp4 — VLM frame-window path) ===')
r = run('ingest_video', {
    'sources':                [SILENT_TEST],
    'hierarchical':           True,
    'enable_frame_captioning': False,  # per-frame captions off; window captions via VLM
    'force_frame_analysis':    False,
    'summarize':               False,
})

assert_ok(r, 'sources_processed')

# Verify routing: silent file must land on the VLM frame path
assert r.get('silent_sources') == 1, \
    f'Expected silent_sources=1, got {r.get("silent_sources")}'
assert r.get('audio_sources')  == 0, \
    f'Expected audio_sources=0, got {r.get("audio_sources")}'

print(f'\nChunks created  : {r.get("chunks_created")}')
print(f'Audio sources   : {r.get("audio_sources")}')
print(f'Silent sources  : {r.get("silent_sources")}')
print(f'Errors          : {r.get("errors")}')
print('[ASSERT OK] silent_sources==1, audio_sources==0')

In [ ]:
%%time
# Smoke-test force_frame_analysis flag: route an audio video through the VLM path
print('=== ingest_video (audio_test.mp4, force_frame_analysis=True) ===')
r = run('ingest_video', {
    'sources':             [AUDIO_TEST],
    'hierarchical':        False,
    'force_frame_analysis': True,   # override: treat as silent even though audio exists
    'summarize':            False,
})
assert_ok(r)
print(f'Silent sources (forced): {r.get("silent_sources")}  -- expected 1')

## 11  `search_knowledge` across video content

Runs hybrid and BM25 searches against the ingested chunks.
Each result row highlights the `content_type` metadata field
(`transcript` vs `frame_window_captions`) so you can see which ingestion
path produced the hit.

In [ ]:
%%time
# Hybrid search — combines dense vector + BM25 scores
print('=== search_knowledge: hybrid ===')
r = run('search_knowledge', {
    'query': 'colour bars calibration visual test pattern',
    'mode':  'hybrid',
    'top_k': 6,
})
assert_ok(r)

if isinstance(r, dict) and r.get('results'):
    print()
    print(f'  {"content_type":<22}  {"source":<22}  {"t/window":<10}  preview')
    print('  ' + '-' * 80)
    for hit in r['results'][:6]:
        meta    = hit.get('metadata', {})
        ctype   = meta.get('content_type', '?')
        ts      = meta.get('start_time', meta.get('window_index', '?'))
        source  = Path(meta.get('source', '?')).name
        preview = hit.get('content', '')[:80].replace('\n', ' ')
        print(f'  {ctype:<22}  {source:<22}  {str(ts):<10}  {preview}...')

In [ ]:
%%time
# BM25-only search for a term that appears in transcript/caption text
print('=== search_knowledge: bm25 ===')
r = run('search_knowledge', {
    'query': 'frame display colour random pattern',
    'mode':  'bm25',
    'top_k': 5,
})
assert_ok(r)

if isinstance(r, dict) and r.get('results'):
    print()
    for hit in r['results'][:4]:
        meta  = hit.get('metadata', {})
        ctype = meta.get('content_type', '?')
        score = hit.get('score', '?')
        print(f'  [{ctype}]  score={score:.4f}  {hit.get("content", "")[:90]}')

## 12  `query_knowledge` grounded in video content  `[LLM]`

> **Requires a configured LLM backend and a valid `OLLAMA_API_KEY`.**

Sends a natural-language question to the RAG pipeline.
The retrieval context is drawn from the ingested video chunks,
exercising both transcript and frame-caption content types.

In [ ]:
%%time
if not skip_llm():
    # Query that should pull hits from both the audio (transcript) and silent (captions) video
    print('=== query_knowledge (video-grounded) ===')
    r = run('query_knowledge', {
        'question': (
            'What types of visual content are present in these videos? '
            'Describe any colour patterns, test signals, or visual features '
            'that appear across different time windows.'
        ),
        'mode': 'hybrid',
    })
    assert_ok(r, 'answer')
    print()
    print('Answer:')
    print('-' * 60)
    print(r['answer'][:900])
    print('-' * 60)
    print(f'Warnings : {r.get("warnings", [])}')

## 13  `VideoSummarizationAgent` — `brief` summary  `[LLM]`

> **Requires a configured LLM backend and VLM.**

`summary_detail='brief'`:
- Window captions: 3–5 focused sentences each
- Chapter summary: 3–5 sentences (1 paragraph)
- Overall summary: 1 short paragraph

In [ ]:
%%time
if not skip_llm():
    from radiant_rag_mcp.app import create_app
    from radiant_rag_mcp.agents.video_summarization import VideoSummarizationAgent
    from radiant_rag_mcp.config import VideoSummarizationConfig

    # Build app and extract the shared LLM client once; reused in sections 14-16
    _app = create_app(CONFIG_PATH)
    _llm = _app._orchestrator._llm

    # Retrieve all silent-video chunks from the vector store
    hits   = _app.search('silent_test', mode='dense', top_k=30, show_results=False)
    chunks = [
        doc for doc, _ in hits
        if Path(doc.meta.get('source', '')).name == 'silent_test.mp4'
    ]
    print(f'Chunks retrieved for silent_test.mp4: {len(chunks)}')

    if not chunks:
        print('[NOTE] No chunks found — run section 10 first.')
    else:
        cfg   = VideoSummarizationConfig(summary_detail='brief')
        agent = VideoSummarizationAgent(llm=_llm, config=cfg)
        _res_brief = agent.summarize_video(SILENT_TEST, chunks)

        print(f'Title      : {_res_brief.title}')
        print(f'is_silent  : {_res_brief.is_silent}')
        print(f'Duration   : {_res_brief.duration_seconds:.0f}s')
        print(f'Chapters   : {len(_res_brief.chapters)}')
        print(f'Key topics : {_res_brief.key_topics}')
        print()
        print('Overall summary (brief):')
        print('-' * 60)
        print(_res_brief.summary)
        print('-' * 60)
        print(f'Word count (overall) : {len(_res_brief.summary.split())}')

## 14  `VideoSummarizationAgent` — `standard` summary  `[LLM]`

> **Requires a configured LLM backend and VLM.**

`summary_detail='standard'`:
- Chapter summary: 1–2 structured paragraphs (5–10 sentences)
- Overall summary: 2–3 paragraphs (Overview + Content + optional Detail)

In [ ]:
%%time
if not skip_llm():
    cfg   = VideoSummarizationConfig(summary_detail='standard')
    agent = VideoSummarizationAgent(llm=_llm, config=cfg)
    _res_standard = agent.summarize_video(SILENT_TEST, chunks)

    print('Overall summary (standard):')
    print('-' * 60)
    print(_res_standard.summary)
    print('-' * 60)
    print(f'Word count (overall) : {len(_res_standard.summary.split())}')
    print()
    print('Chapter breakdowns:')
    for ch in _res_standard.chapters:
        print(f'  Ch {ch.index + 1}: [{ch.start:.0f}s – {ch.end:.0f}s]  {ch.title}')
        for ln in ch.summary.split('\n')[:4]:
            print(f'    {ln[:110]}')
        print()

## 15  `VideoSummarizationAgent` — `detailed` summary  `[LLM]`

> **Requires a configured LLM backend and VLM.**

`summary_detail='detailed'`:
- Chapter summary: 2–3 paragraphs (8–15 sentences)
- Overall summary: 3–4 structured paragraphs
  (Overview / Content / Detail / Takeaway schema)

In [ ]:
%%time
if not skip_llm():
    cfg   = VideoSummarizationConfig(summary_detail='detailed')
    agent = VideoSummarizationAgent(llm=_llm, config=cfg)
    _res_detailed = agent.summarize_video(SILENT_TEST, chunks)

    print('Overall summary (detailed):')
    print('-' * 60)
    print(_res_detailed.summary)
    print('-' * 60)
    print(f'Word count (overall) : {len(_res_detailed.summary.split())}')
    print()
    print(f'Chapters : {len(_res_detailed.chapters)}')
    for ch in _res_detailed.chapters:
        print(f'  Ch {ch.index + 1}: [{ch.start:.0f}s – {ch.end:.0f}s]  {ch.title}')

## 16  Summary detail levels — word-count comparison  `[LLM]`

> **Requires a configured LLM backend and VLM.**

Runs all three detail levels and prints a side-by-side table covering:
- **Window** average word count (across all window captions, where available)
- **Chapter** average word count
- **Overall** total word count

Expected trend: `brief` < `standard` < `detailed` in every column.

In [ ]:
%%time
if not skip_llm():
    # Collect results for all three detail levels
    # Reuse cached results from sections 13-15 if they exist; otherwise re-run
    detail_results = {}
    for detail, cached in [
        ('brief',    globals().get('_res_brief')),
        ('standard', globals().get('_res_standard')),
        ('detailed', globals().get('_res_detailed')),
    ]:
        if cached is not None:
            detail_results[detail] = cached
            print(f'  {detail:<10}  (using cached result from section 13/14/15)')
        else:
            cfg   = VideoSummarizationConfig(summary_detail=detail)
            agent = VideoSummarizationAgent(llm=_llm, config=cfg)
            detail_results[detail] = agent.summarize_video(SILENT_TEST, chunks)
            print(f'  {detail:<10}  (freshly generated)')

    print()
    # Build comparison table: window / chapter / overall word counts
    HDR = f'  {"Detail":<10}  {"Window avg":>12}  {"Chapter avg":>12}  {"Overall total":>14}  {"Chapters":>8}'
    SEP = '  ' + '-' * (len(HDR) - 2)
    print(HDR)
    print(SEP)

    for detail, res in detail_results.items():
        # Overall summary word count
        overall_wc = len(res.summary.split())

        # Chapter average word count
        ch_wcs = [len(ch.summary.split()) for ch in res.chapters]
        ch_avg = (sum(ch_wcs) // max(1, len(ch_wcs))) if ch_wcs else 0

        # Window average word count — use window_captions if present, else chapter captions
        win_wcs = []
        if hasattr(res, 'window_captions') and res.window_captions:
            win_wcs = [len(w.caption.split()) for w in res.window_captions if hasattr(w, 'caption')]
        elif res.chapters:
            # Fall back: treat chapter summaries as window proxies
            win_wcs = ch_wcs
        win_avg = (sum(win_wcs) // max(1, len(win_wcs))) if win_wcs else 0

        print(f'  {detail:<10}  {win_avg:>12,}  {ch_avg:>12,}  {overall_wc:>14,}  {len(res.chapters):>8}')

    print()
    print('[NOTE] Window avg falls back to chapter avg when per-window captions are unavailable.')

## 17  YouTube URL ingestion  `[LLM]`

> **Requires a configured LLM backend and VLM.**

Downloads a short public YouTube video via `yt-dlp`, ingests it through the
MCP tool, and searches the resulting transcript chunks.
The video has audio, so it travels the Whisper transcription path.

Target: [3Blue1Brown — "But what is a neural network?"](https://www.youtube.com/watch?v=aircAruvnKk)
(≈ 19 min; only the first few minutes are transcribed during the default window budget).

> **Runtime:** ~60–120 s (yt-dlp download + Whisper base).

In [ ]:
%%time
if not skip_llm():
    YT_URL = 'https://www.youtube.com/watch?v=aircAruvnKk'

    # Ingest the YouTube URL directly; the tool calls yt-dlp internally
    print(f'=== ingest_video (YouTube URL) ===')
    print(f'URL: {YT_URL}')
    r = run('ingest_video', {
        'sources':    [YT_URL],
        'hierarchical': True,
        'summarize':    True,   # generate a structured summary after ingestion
    })
    assert_ok(r, 'sources_processed')

    print(f'\nChunks created : {r.get("chunks_created")}')
    print(f'Audio sources  : {r.get("audio_sources")}')
    print(f'Silent sources : {r.get("silent_sources")}')
    print(f'Errors         : {r.get("errors")}')

    # Display generated summary for each ingested source
    for src, s in r.get('summaries', {}).items():
        print(f'\n--- Summary for: {src} ---')
        print(f'  Title     : {s.get("title")}')
        print(f'  is_silent : {s.get("is_silent")}')
        print(f'  Chapters  : {len(s.get("chapters", []))}')
        print()
        print(s.get('summary', '')[:700])

In [ ]:
%%time
if not skip_llm():
    # Search the YouTube transcript chunks to confirm they are indexed
    print('=== search_knowledge (YouTube transcript) ===')
    r = run('search_knowledge', {
        'query': 'neural network layers neurons activation',
        'mode':  'hybrid',
        'top_k': 4,
    })
    assert_ok(r)
    if isinstance(r, dict) and r.get('results'):
        for hit in r['results'][:3]:
            meta  = hit.get('metadata', {})
            ctype = meta.get('content_type', '?')
            src   = Path(meta.get('source', '?')).name[:40]
            print(f'  [{ctype}]  {src}')
            print(f'    {hit.get("content", "")[:100]}')

## 18  Cleanup

- Clears the ChromaDB vector index
- Deletes the temporary MP4 files from `/tmp/radiant_video_test/`

Skip the `clear_index` call if you want to keep the chunks for further
experimentation; set `RUN_CLEAR = False` in the cell below.

In [ ]:
RUN_CLEAR = True   # Set to False to keep the index intact

if RUN_CLEAR:
    print('=== clear_index ===')
    r = run('clear_index', {'confirm': True})
    if isinstance(r, dict):
        if r.get('cleared'):
            print('[OK] Index cleared.')
        elif ('_ensure_index' in r.get('message', '')
              or 'clear failed' in r.get('message', '').lower()):
            # Known ChromaDB reinit issue — collection was dropped
            print('[OK] Collection dropped (known ChromaDB reinit pattern).')
        else:
            raise AssertionError(f'Unexpected clear failure: {r}')

    # Confirm document count is 0
    stats = asyncio.run(_call('get_index_stats'))
    if isinstance(stats, dict):
        vi = stats.get('health', stats).get('vector_index', {})
        print(f'Vector doc count post-clear: {vi.get("document_count", "?")}')
else:
    print('RUN_CLEAR=False — index preserved.')

In [ ]:
import shutil

# Delete temporary video files
_test_dir = Path('/tmp/radiant_video_test')
if _test_dir.exists():
    shutil.rmtree(_test_dir)
    print(f'Removed test video directory: {_test_dir}')
else:
    print(f'Directory not found (already cleaned?): {_test_dir}')

print('Cleanup complete')

## Summary

Run this cell at any point for a status overview.

In [ ]:
print('=' * 65)
print('  Radiant RAG MCP — Video Ingestion & Summarization Test')
print('  Run complete')
print('=' * 65)
print()
print('  Synthetic test videos  (640×480, 30 s)')
print('    audio_test.mp4   — SMPTE colour bars + 1 kHz sine tone')
print('    silent_test.mp4  — random solid-colour frames, no audio')
print()
print('  Audio detection')
print('    ok  _has_audio(audio_test.mp4)  == True')
print('    ok  _has_audio(silent_test.mp4) == False')
print()
print('  Video ingestion')
print('    ok  audio_test.mp4   -> Whisper transcription -> transcript chunks')
print('    ok  silent_test.mp4  -> VLM frame windows     -> caption chunks')
print('    ok  force_frame_analysis flag override')
print()
print('  Search')
print('    ok  search_knowledge hybrid  (content_type highlighted)')
print('    ok  search_knowledge bm25')
print()
print('  LLM / VLM cells  [L = requires OLLAMA_API_KEY]')
print('   [L]  query_knowledge grounded in video content')
print('   [L]  VideoSummarizationAgent  brief')
print('   [L]  VideoSummarizationAgent  standard')
print('   [L]  VideoSummarizationAgent  detailed')
print('   [L]  Word-count table  (window / chapter / overall)')
print('   [L]  YouTube URL ingestion + transcript search')
print()
print(f'  LLM key available : {HAS_LLM_KEY}')